# 🐾 Animal Classification — ConvNeXt + Optuna + Cosine Decay

**Upgrades over baseline:**
- ✅ ConvNeXt-Base (pretrained) replacing AlexNet — stronger inductive bias for shapes
- ✅ Layer-wise LR decay for ConvNeXt finetuning
- ✅ Aggressive color augmentation (ColorJitter, RandomGrayscale, hue shift)
- ✅ Optuna hyperparameter tuning for LR
- ✅ Cosine annealing LR scheduler
- ✅ Stratified train/val split
- ✅ Per-epoch logging: loss + macro F1 on val
- ✅ Best checkpoint saving
- ✅ Final evaluation with per-class F1 + confusion matrix
- ✅ Training curves visualization
- ✅ Submission file named `submission_{f1_score}.csv`
- ✅ All weights zipped for download

In [ ]:
# ─── Install dependencies ───────────────────────────────────────────────────
!pip install -q optuna timm

In [ ]:
# ─── Imports ────────────────────────────────────────────────────────────────
import os
import random
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix
)

# ─── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── Config ─────────────────────────────────────────────────────────────────
class CFG:
    MODEL_NAME   = 'convnext_base'      # timm model name
    IMG_SIZE     = 224          # convnext_base native; use 256 for boost
    LAYER_DECAY  = 0.75         # layer-wise LR decay factor (ConvNeXt best practice)
    BATCH_SIZE   = 32
    NUM_EPOCHS   = 20                  # full training epochs after Optuna
    OPTUNA_TRIALS= 15                  # number of Optuna trials
    OPTUNA_EPOCHS= 5                   # epochs per Optuna trial (fast)
    NUM_WORKERS  = 2
    LABEL_SMOOTH = 0.1
    CHECKPOINT_DIR = '/kaggle/working/checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Config loaded ✓')

## 1. Load Data

In [ ]:
def find_file(name):
    for root, dirs, files in os.walk('/kaggle/input'):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

train_csv  = find_file('train.csv')
sample_csv = find_file('sample_submission.csv')
base_dir   = os.path.dirname(train_csv)
train_dir  = os.path.join(base_dir, 'train_images')
test_dir   = os.path.join(base_dir, 'test_images')

train_df      = pd.read_csv(train_csv)
submission_df = pd.read_csv(sample_csv)

# ─── Label encoding ─────────────────────────────────────────────────────────
classes      = sorted(train_df['animal_class'].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
train_df['label'] = train_df['animal_class'].map(class_to_idx)

print(f'Training images : {len(train_df)}')
print(f'Test images     : {len(submission_df)}')
print(f'Classes         : {len(classes)}')

In [ ]:
# ─── Class distribution ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 5))
counts = train_df['animal_class'].value_counts()
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Class Distribution in Training Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Animal Class')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150)
plt.show()
print(f'Min samples: {counts.min()} | Max samples: {counts.max()}')

## 2. Augmentation Strategy

> **Key insight:** Colors are meaningless in this dataset (same animal = different colors).
> We aggressively randomize hue/saturation/brightness to force the model to learn **shape**, not color.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE + 32, CFG.IMG_SIZE + 32)),
    transforms.RandomCrop(CFG.IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=20),

    # ── Color-invariance augmentations (CRITICAL for this dataset) ──
    transforms.ColorJitter(
        brightness=0.6,
        contrast=0.6,
        saturation=0.6,
        hue=0.5           # full hue randomization
    ),
    transforms.RandomGrayscale(p=0.25),   # forces shape learning
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
    ], p=0.2),

    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),  # cutout-style
])

val_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── TTA (Test Time Augmentation) transforms ──────────────────────────────
tta_transforms = [
    val_transform,
    transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]

print('Transforms defined ✓')

## 3. Dataset & Dataloader

In [ ]:
class AnimalDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(
            os.path.join(self.image_dir, row['image_id'])
        ).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label


class TestDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.iloc[idx]['image_id']
        image    = Image.open(
            os.path.join(self.image_dir, image_id)
        ).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, image_id


# ─── Stratified split ───────────────────────────────────────────────────────
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(sss.split(train_df, train_df['label']))

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

# ─── Weighted sampler for class imbalance ──────────────────────────────────
label_counts  = train_split['label'].value_counts().sort_index()
class_weights = 1.0 / label_counts.values
sample_weights = np.array([class_weights[l] for l in train_split['label']])
sampler = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_weights),
    num_samples=len(train_split),
    replacement=True
)

def get_loaders(batch_size=CFG.BATCH_SIZE):
    train_ds = AnimalDataset(train_split, train_dir, train_transform)
    val_ds   = AnimalDataset(val_split,   train_dir, val_transform)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True
    )
    return train_loader, val_loader

print(f'Train split : {len(train_split)}')
print(f'Val split   : {len(val_split)}')

## 4. Model

In [ ]:
def build_model(num_classes=len(classes), dropout=0.3):
    model = timm.create_model(
        CFG.MODEL_NAME,
        pretrained=True,
        num_classes=num_classes,
        drop_rate=dropout
    )
    return model.to(device)

# ─── Label smoothing loss ───────────────────────────────────────────────────
loss_fn = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTH)

print(f'Model: {CFG.MODEL_NAME} ✓')

def build_optimizer_with_layer_decay(model, lr, weight_decay, layer_decay=CFG.LAYER_DECAY):
    """
    Layer-wise LR decay for ConvNeXt — deeper layers get lower LR.
    Significantly improves finetuning stability on pretrained models.
    """
    param_groups = []
    num_layers = len(list(model.named_parameters()))
    
    for idx, (name, param) in enumerate(model.named_parameters()):
        if not param.requires_grad:
            continue
        # Compute layer depth-based decay
        depth_ratio = idx / num_layers
        layer_lr = lr * (layer_decay ** (1.0 - depth_ratio))
        
        # No weight decay on bias / norm params
        wd = 0.0 if ('bias' in name or 'norm' in name or 'bn' in name) else weight_decay
        param_groups.append({'params': [param], 'lr': layer_lr, 'weight_decay': wd})
    
    return torch.optim.AdamW(param_groups)


## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    if scheduler:
        scheduler.step()
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = loss_fn(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1, all_labels, all_preds


print('Training utilities defined ✓')

## 6. Optuna — LR Search

In [ ]:
def optuna_objective(trial):
    lr      = trial.suggest_float('lr',      1e-5, 1e-2, log=True)
    wd      = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)

    model     = build_model(dropout=dropout)
    optimizer = build_optimizer_with_layer_decay(model, lr=lr, weight_decay=wd, layer_decay=CFG.LAYER_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.OPTUNA_EPOCHS
    )
    train_loader, val_loader = get_loaders(batch_size=32)

    for epoch in range(CFG.OPTUNA_EPOCHS):
        train_one_epoch(model, train_loader, optimizer, scheduler)
        val_loss, val_f1, _, _ = evaluate(model, val_loader)
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    del model
    torch.cuda.empty_cache()
    return val_f1


print('Starting Optuna search...')
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2)
)
study.optimize(
    optuna_objective,
    n_trials=CFG.OPTUNA_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params
print(f'\nBest trial macro-F1 : {study.best_value:.4f}')
print(f'Best params         : {best_params}')

In [ ]:
# ─── Optuna visualizations ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Optimization history
values = [t.value for t in study.trials if t.value is not None]
axes[0].plot(values, marker='o', color='royalblue')
axes[0].axhline(max(values), color='red', linestyle='--', alpha=0.6, label=f'Best={max(values):.4f}')
axes[0].set_title('Optuna — Optimization History', fontweight='bold')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Macro F1 (val)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# LR vs F1 scatter
lrs = [t.params['lr'] for t in study.trials if t.value is not None]
f1s = [t.value for t in study.trials if t.value is not None]
axes[1].scatter(lrs, f1s, c=f1s, cmap='viridis', s=80, edgecolors='k', lw=0.5)
axes[1].set_xscale('log')
axes[1].set_title('LR vs Val Macro F1', fontweight='bold')
axes[1].set_xlabel('Learning Rate (log scale)')
axes[1].set_ylabel('Macro F1')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/optuna_results.png', dpi=150)
plt.show()

## 7. Full Training with Best Hyperparameters

In [ ]:
# ─── Init model with best Optuna params ─────────────────────────────────────
BEST_LR      = best_params['lr']
BEST_WD      = best_params['weight_decay']
BEST_DROPOUT = best_params['dropout']

model     = build_model(dropout=BEST_DROPOUT)
optimizer = build_optimizer_with_layer_decay(model, lr=BEST_LR, weight_decay=BEST_WD, layer_decay=CFG.LAYER_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.NUM_EPOCHS, eta_min=BEST_LR * 0.01
)
train_loader, val_loader = get_loaders(CFG.BATCH_SIZE)

# ─── Logging containers ─────────────────────────────────────────────────────
history = {
    'epoch':      [],
    'train_loss': [],
    'val_loss':   [],
    'val_f1':     [],
    'lr':         []
}
best_val_f1     = 0.0
best_epoch      = 0
best_ckpt_path  = os.path.join(CFG.CHECKPOINT_DIR, 'best_model.pt')

print(f'Training for {CFG.NUM_EPOCHS} epochs | LR={BEST_LR:.2e} | WD={BEST_WD:.2e}')
print('─' * 65)
print(f'{"Epoch":>6} | {"Train Loss":>11} | {"Val Loss":>9} | {"Val F1":>8} | {"LR":>10}')
print('─' * 65)

In [ ]:
for epoch in range(1, CFG.NUM_EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']

    train_loss             = train_one_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_f1, _, _ = evaluate(model, val_loader)

    # ── Log ──────────────────────────────────────────────────────────────
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)

    print(f'{epoch:>6} | {train_loss:>11.4f} | {val_loss:>9.4f} | {val_f1:>8.4f} | {current_lr:>10.2e}')

    # ── Checkpoint if best ────────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        torch.save({
            'epoch':      epoch,
            'model_state_dict':    model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_f1':     val_f1,
            'val_loss':   val_loss,
            'best_params': best_params,
            'class_to_idx': class_to_idx,
            'idx_to_class': idx_to_class,
        }, best_ckpt_path)
        print(f'         ↑ New best! Checkpoint saved (F1={val_f1:.4f})')

    # ── Per-epoch checkpoint ──────────────────────────────────────────────
    epoch_ckpt = os.path.join(CFG.CHECKPOINT_DIR, f'epoch_{epoch:02d}_f1_{val_f1:.4f}.pt')
    torch.save(model.state_dict(), epoch_ckpt)

print('─' * 65)
print(f'Training complete! Best val F1 = {best_val_f1:.4f} @ epoch {best_epoch}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Training Summary — {CFG.MODEL_NAME}', fontsize=14, fontweight='bold')

epochs = history['epoch']

# ── Loss ──────────────────────────────────────────────────────────────────
axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='royalblue', marker='o', ms=4)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='tomato',    marker='s', ms=4)
axes[0].axvline(best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best epoch={best_epoch}')
axes[0].set_title('Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Macro F1 ───────────────────────────────────────────────────────────────
axes[1].plot(epochs, history['val_f1'], label='Val Macro F1', color='seagreen', marker='D', ms=4)
axes[1].axhline(best_val_f1, color='red', linestyle='--', alpha=0.6, label=f'Best F1={best_val_f1:.4f}')
axes[1].axvline(best_epoch, color='green', linestyle='--', alpha=0.7)
axes[1].set_title('Validation Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── LR Schedule ────────────────────────────────────────────────────────────
axes[2].plot(epochs, history['lr'], color='darkorange', marker='^', ms=4)
axes[2].set_title('Learning Rate (Cosine Annealing)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

## 9. Final Evaluation on Val Set

In [ ]:
# ─── Load best checkpoint ───────────────────────────────────────────────────
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]} (val F1={ckpt["val_f1"]:.4f})')

_, final_f1, true_labels, pred_labels = evaluate(model, val_loader)
print(f'\nFinal Val Macro F1: {final_f1:.4f}')

In [ ]:
# ─── Per-class F1 ───────────────────────────────────────────────────────────
report = classification_report(
    true_labels, pred_labels,
    target_names=classes,
    output_dict=True,
    zero_division=0
)
per_class_f1 = pd.DataFrame(report).T.loc[classes, 'f1-score'].sort_values()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Per-class F1 bar chart
colors = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.75 else '#2ecc71' for v in per_class_f1.values]
per_class_f1.plot(kind='barh', ax=axes[0], color=colors, edgecolor='white')
axes[0].axvline(final_f1, color='navy', linestyle='--', label=f'Macro F1={final_f1:.4f}')
axes[0].set_title('Per-class F1 Score', fontweight='bold')
axes[0].set_xlabel('F1 Score')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Top-10 worst classes
worst_10 = per_class_f1.head(10)
worst_10.plot(kind='barh', ax=axes[1], color='#e74c3c', edgecolor='white')
axes[1].set_title('10 Worst Predicted Classes', fontweight='bold')
axes[1].set_xlabel('F1 Score')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/per_class_f1.png', dpi=150)
plt.show()

print(classification_report(true_labels, pred_labels, target_names=classes, zero_division=0))

In [ ]:
# ─── Confusion matrix (top 30 confused classes) ─────────────────────────────
cm = confusion_matrix(true_labels, pred_labels)

# Normalize
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(
    cm_norm,
    xticklabels=classes,
    yticklabels=classes,
    cmap='Blues',
    ax=ax,
    vmin=0, vmax=1,
    linewidths=0.3,
    cbar_kws={'label': 'Normalized count'}
)
ax.set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

## 10. TTA Inference on Test Set

In [ ]:
@torch.no_grad()
def predict_with_tta(model, df, image_dir, tta_transforms):
    """Average softmax probabilities across all TTA transforms."""
    model.eval()
    all_probs = None
    image_ids = []

    for t_idx, tfm in enumerate(tta_transforms):
        test_ds     = TestDataset(df, image_dir, transform=tfm)
        test_loader = DataLoader(
            test_ds,
            batch_size=CFG.BATCH_SIZE * 2,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS
        )
        probs_list = []
        ids_list   = []

        for images, img_ids in test_loader:
            images  = images.to(device)
            outputs = model(images)
            probs   = F.softmax(outputs, dim=1)
            probs_list.append(probs.cpu().numpy())
            if t_idx == 0:
                ids_list.extend(img_ids)

        batch_probs = np.concatenate(probs_list, axis=0)
        if all_probs is None:
            all_probs  = batch_probs
            image_ids  = ids_list
        else:
            all_probs += batch_probs
        print(f'TTA pass {t_idx+1}/{len(tta_transforms)} done')

    all_probs /= len(tta_transforms)
    pred_indices = all_probs.argmax(axis=1)
    predictions  = [idx_to_class[i] for i in pred_indices]
    return image_ids, predictions


print('Running TTA inference on test set...')
image_ids, predictions = predict_with_tta(model, submission_df, test_dir, tta_transforms)
print(f'Predictions: {len(predictions)}')

## 11. Submission

In [ ]:
# ─── Save submission named with final F1 ────────────────────────────────────
f1_str       = f'{final_f1:.4f}'
sub_filename = f'submission_{f1_str}.csv'
sub_path     = f'/kaggle/working/{sub_filename}'

submission = pd.DataFrame({
    'image_id':    image_ids,
    'animal_class': predictions
})
submission.to_csv(sub_path, index=False)

print(f'Submission saved: {sub_filename}')
print(submission.head(10))
print(f'\nPrediction distribution:')
print(submission['animal_class'].value_counts())

## 12. Zip All Checkpoints

In [ ]:
zip_path = f'/kaggle/working/checkpoints_f1_{f1_str}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model
    zf.write(best_ckpt_path, arcname='best_model.pt')
    # All epoch checkpoints
    for fname in os.listdir(CFG.CHECKPOINT_DIR):
        fpath = os.path.join(CFG.CHECKPOINT_DIR, fname)
        zf.write(fpath, arcname=f'epochs/{fname}')
    # Training artifacts
    for artifact in ['training_curves.png', 'per_class_f1.png',
                     'confusion_matrix.png', 'optuna_results.png',
                     'class_distribution.png', sub_filename]:
        artifact_path = f'/kaggle/working/{artifact}'
        if os.path.exists(artifact_path):
            zf.write(artifact_path, arcname=artifact)

zip_size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'Checkpoint zip saved: {os.path.basename(zip_path)} ({zip_size_mb:.1f} MB)')
print('\nZip contents:')
with zipfile.ZipFile(zip_path, 'r') as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f'  {name:55s}  {info.file_size/1024:.0f} KB')

## 13. Summary

In [ ]:
print('=' * 60)
print('           EXPERIMENT SUMMARY')
print('=' * 60)
print(f'  Model             : {CFG.MODEL_NAME}')
print(f'  Best Optuna F1    : {study.best_value:.4f}')
print(f'  Best LR           : {BEST_LR:.2e}')
print(f'  Best WD           : {BEST_WD:.2e}')
print(f'  Best Dropout      : {BEST_DROPOUT:.2f}')
print(f'  Training Epochs   : {CFG.NUM_EPOCHS}')
print(f'  Best Epoch        : {best_epoch}')
print(f'  Final Val F1      : {final_f1:.4f}')
print(f'  Submission file   : {sub_filename}')
print(f'  Checkpoint zip    : {os.path.basename(zip_path)}')
print('=' * 60)